In [1]:
# Uncomment and run this command and subsequent ones if they are not present on your system!
# !pip install torch

In [2]:
# !pip install torchvision

In [3]:
# !pip install pillow

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

In [5]:
# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
imsize = 512 if torch.cuda.is_available() else 256  # Resize if using CPU

## Neural Style Transfer

This is a PyTorch implementation of the method that was discussed in Gatys. et Al.

In [6]:
# Image Loading and Preprocessing
loader = transforms.Compose([
    transforms.Resize((imsize, imsize)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def load_image(image_name):
    image = Image.open(image_name).convert('RGB')
    image = loader(image).unsqueeze(0)
    return image.to(device, torch.float)

# Load your local images here
content_img = load_image("RoadatNTU.jpeg")
style_img = load_image("TheSailor.jpg")

# Initialize target image as a copy of content image, requiring gradients
target_img = content_img.clone().detach().requires_grad_(True)

In [7]:
# Setting up the loss layers
class ContentLoss(nn.Module):
    def __init__(self, target):
        super(ContentLoss, self).__init__()
        self.target = target.detach()

    def forward(self, x):
        self.loss = nn.functional.mse_loss(x, self.target)
        return x

def gram_matrix(x):
    batch, channels, height, width = x.size()
    features = x.view(batch * channels, height * width)
    G = torch.mm(features, features.t())
    return G.div(batch * channels * height * width)

class StyleLoss(nn.Module):
    def __init__(self, target_feature):
        super(StyleLoss, self).__init__()
        self.target = gram_matrix(target_feature).detach()

    def forward(self, x):
        G = gram_matrix(x)
        self.loss = nn.functional.mse_loss(G, self.target)
        return x

In [8]:
# Preparing the network
# VGG-19 is used here as it was the same layered network that was used in the original
vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(device).eval()

content_layers = ['conv4_2']
style_layers = ['conv1_1', 'conv2_1', 'conv3_1', 'conv4_1', 'conv5_1']

content_losses = []
style_losses = []

model = nn.Sequential()
block_id = 0
conv_id = 0
for layer in vgg.children():
    if isinstance(layer, nn.Conv2d):
        name = f'conv{block_id}_{conv_id}'
        conv_id += 1
    elif isinstance(layer, nn.ReLU):
        name = f'relu{block_id}_{conv_id-1}'
        layer = nn.ReLU(inplace=False) 
    elif isinstance(layer, nn.MaxPool2d):
        name = f'pool_{block_id}'
        block_id += 1  
        conv_id = 1     
    else:
        name = f'layer_{block_id}_{conv_id}'

    model.add_module(name, layer)

    if name in content_layers:
        target_feature = model(content_img).detach()
        content_loss = ContentLoss(target_feature)
        model.add_module(f"content_loss_{block_id}_{conv_id - 1}", content_loss)
        content_losses.append(content_loss)

    if name in style_layers:
        target_feature = model(style_img).detach()
        style_loss = StyleLoss(target_feature)
        model.add_module(f"style_loss_{block_id}_{conv_id - 1}", style_loss)
        style_losses.append(style_loss)

# Truncate network after the last loss layer to save computing time
for i in range(len(model) - 1, -1, -1):
    if isinstance(model[i], (ContentLoss, StyleLoss)):
        model = model[:(i + 1)]
        break


In [9]:
# Optimization Parameters
optimizer = optim.LBFGS([target_img]) 
style_weight = 1000000 # Original, untuned version
# tyle_weight = 10000

content_weight = 1 # Original, untuned version
# content_weight = 1000 

In [10]:
# Execution Loop
run = [0]
num_of_iterations = 400

start_time = time.perf_counter()
while run[0] <= num_of_iterations:
    def closure():

        optimizer.zero_grad()
        model(target_img)
        
        style_score = torch.tensor(0.0, device=device)
        content_score = torch.tensor(0.0, device=device)

        for sl in style_losses:
            style_score += sl.loss
        for cl in content_losses:
            content_score += cl.loss

        style_score *= style_weight
        content_score *= content_weight
        loss = style_score + content_score
        loss.backward()

        run[0] += 1
        if run[0] % 50 == 0:
            print(f"Step {run[0]}: Style Loss: {style_score.item():.4f} Content Loss: {content_score.item():.4f}")
        return loss

    optimizer.step(closure)
end_time = time.perf_counter()

# Calculate elapsed time
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.6f} seconds")

Step 50: Style Loss: 172.0895 Content Loss: 13.2287
Step 100: Style Loss: 58.6573 Content Loss: 12.4366
Step 150: Style Loss: 29.9107 Content Loss: 11.4062
Step 200: Style Loss: 20.4431 Content Loss: 10.6041
Step 250: Style Loss: 15.6393 Content Loss: 10.0134
Step 300: Style Loss: 12.8522 Content Loss: 9.6024
Step 350: Style Loss: 11.0025 Content Loss: 9.2506
Step 400: Style Loss: 9.7121 Content Loss: 8.9978
Execution time: 100.778265 seconds


In [11]:
# Deprocess and Save Output
with torch.no_grad():
    target_img.clamp_(0, 1)

# Inverse Normalize to save as standard image
inv_normalize = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)
start_time = time.perf_counter()
output_tensor = inv_normalize(target_img.squeeze(0))
output_image = transforms.ToPILImage()(output_tensor.clamp(0, 1))
output_image.save("stylized_output.jpg")
print("Stylized image saved successfully!")
end_time = time.perf_counter()

# Calculate elapsed time
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.6f} seconds")

Stylized image saved successfully!
Execution time: 0.043275 seconds
